# Bronze → Silver
Cleans stats and events for one game and builds the player mapping. Output: Parquet in `data/silver/`.

In [ ]:
from pathlib import Path

import yaml
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import NumericType

from utility import (q, find_bronze_file, read_bronze_csv,
                     clean_stats, clean_events, build_players)

spark = SparkSession.builder.master("local[*]").appName("bronze_to_silver").getOrCreate()

# Parameters & Helper Functions

In [ ]:
cfg = yaml.safe_load(open("../config/keyplayer_battle.yaml", encoding="utf-8"))

DATA = "../" + cfg["data"]["root"]
YEAR = cfg["active_game"]["year"]
GAME = cfg["active_game"]["game"]
print(f"Processing {YEAR}/{GAME:02d}")

def q(c):                      # column with any name (spaces, dots, %)
    return F.col(f"`{c}`")

def jersey(c):                 # "#18 Firstname Lastname" -> 18
    return F.regexp_extract(q(c), r"^#(\d+)", 1).cast("int")

def to_bool(c):                # "True"/"False" -> real booleans, empty stays empty
    s = F.lower(F.trim(q(c).cast("string")))
    return F.when(s == "true", True).when(s == "false", False)

# Read & Write To Silver 

In [ ]:
def read_bronze(kind):
    folder = Path(DATA) / cfg["data"]["bronze"][kind].format(year=YEAR, game=GAME)
    return read_bronze_csv(spark, find_bronze_file(folder, cfg["data"]["bronze_files"][kind]))

def write_silver(df, kind):
    out = Path(DATA) / cfg["data"]["silver"][kind].format(year=YEAR, game=GAME)
    df.write.mode("overwrite").parquet(str(out))
    print(f"{kind}: {df.count()} rows → {out}")

## Stats and Events To Silver 
Events keeps Home/Away, makes IDs and xG numbers, goalie flags True/False

In [ ]:

st = clean_stats(read_bronze("stats"), YEAR, GAME)
ev = clean_events(read_bronze("events"), YEAR, GAME)

## Player-Mapping
Works out Home/Away to team by itself, then links event IDs to stats

In [ ]:
players = build_players(ev, st, YEAR, GAME)
players.orderBy("Team", "Jersey").select("player_id", "Team", "Jersey", "in_stats").show(50)

## Data checks
Stops the run if the data doesn't match what the KPI functions expect.

In [ ]:
text_cols = ["Team", "LastName", "FirstName", "player_key"]
bad_types = [f.name for f in st.schema.fields if f.name not in text_cols and not isinstance(f.dataType, NumericType)]
assert not bad_types, f"Stats columns not numeric: {bad_types}"

pct_cols = [c for c in st.columns if "%" in c]
pct_max = st.select(*[F.max(F.abs(q(c))).alias(c) for c in pct_cols]).first().asDict()
assert all(v is None or v <= 1 for v in pct_max.values()), f"Percentages not 0–1: {pct_max}"
assert st.select("player_key").distinct().count() == st.count(), "Duplicate team + jersey in stats"

bad_shots = ev.filter((F.col("EventType") == "Shot") &
                      ~F.coalesce(F.col("ShotResult").isin("Saved", "Goal", "Missed", "Blocked"), F.lit(False))).count()
assert bad_shots == 0, f"{bad_shots} shots with unknown ShotResult"

assert players.groupBy("player_id").count().filter("count > 1").count() == 0, "Player ID with several names/teams"
print("All checks passed")

## Save as Parquet

In [ ]:
write_silver(st, "stats")
write_silver(ev, "events")
write_silver(players, "players")